# AI Voice Detection - Dataset Exploration

This notebook explores the dataset and validates the feature extraction pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

import config
from features import LFCCExtractor, MelSpectrogramExtractor, load_audio, pad_or_trim
from models import build_lightweight_cnn, compile_model

## 1. Configuration Overview

In [ ]:
print("Audio Configuration:")
print(f"  Sample Rate: {config.SAMPLE_RATE} Hz")
print(f"  Window Duration: {config.AUDIO_DURATION} seconds")
print(f"  Samples per Window: {config.AUDIO_SAMPLES}")

print("\nFeature Configuration:")
print(f"  LFCC Coefficients: {config.N_LFCC}")
print(f"  FFT Size: {config.N_FFT}")
print(f"  Hop Length: {config.HOP_LENGTH}")
print(f"  Input Shape: {config.INPUT_SHAPE}")

## 2. Feature Extraction Demo

In [ ]:
# Generate synthetic audio for demo
duration = config.AUDIO_DURATION
sr = config.SAMPLE_RATE
t = np.linspace(0, duration, int(sr * duration))

# Create a simple speech-like signal
audio = np.sin(2 * np.pi * 200 * t) * np.exp(-t/2)  # Decaying sine
audio += 0.5 * np.sin(2 * np.pi * 400 * t) * np.exp(-t/1.5)
audio += 0.3 * np.sin(2 * np.pi * 800 * t) * np.exp(-t/1)
audio += 0.1 * np.random.randn(len(audio))  # Add noise

print(f"Audio shape: {audio.shape}")
print(f"Audio duration: {len(audio)/sr:.2f} seconds")

In [ ]:
# Extract LFCC features
lfcc_extractor = LFCCExtractor()
lfcc_features = lfcc_extractor.extract(audio)

print(f"LFCC shape: {lfcc_features.shape}")
print(f"Expected shape: {config.INPUT_SHAPE}")

In [ ]:
# Visualize LFCC
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Waveform
axes[0].plot(t, audio)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Waveform')

# LFCC
im = axes[1].imshow(lfcc_features[:, :, 0], aspect='auto', origin='lower', cmap='viridis')
axes[1].set_xlabel('Time Frames')
axes[1].set_ylabel('LFCC Coefficient')
axes[1].set_title('LFCC Features')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 3. Model Architecture

In [ ]:
# Build and inspect model
model = build_lightweight_cnn()
compile_model(model)

model.summary()

In [ ]:
# Count parameters
total_params = model.count_params()
print(f"Total parameters: {total_params:,}")
print(f"Estimated model size: {total_params * 4 / (1024*1024):.2f} MB (float32)")

In [ ]:
# Test forward pass
batch = np.expand_dims(lfcc_features, axis=0)
prediction = model.predict(batch, verbose=0)

print(f"Input shape: {batch.shape}")
print(f"Output shape: {prediction.shape}")
print(f"Prediction: {prediction[0][0]:.4f}")

## 4. Augmentation Demo

In [ ]:
from data.augmentation import AudioAugmenter, VoIPSimulator

# Create augmenter
augmenter = AudioAugmenter(probability=1.0)  # Apply all augmentations

# Augment audio
augmented = augmenter(audio)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(t, audio)
axes[0].set_title('Original Audio')
axes[0].set_xlabel('Time (s)')

axes[1].plot(t, augmented)
axes[1].set_title('Augmented Audio (VoIP Simulation)')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

## 5. Inference Speed Test

In [ ]:
import time

# Benchmark feature extraction + inference
n_iterations = 100
times = []

for _ in range(n_iterations):
    # Generate random audio
    test_audio = np.random.randn(config.AUDIO_SAMPLES).astype(np.float32)
    
    start = time.perf_counter()
    
    # Feature extraction
    features = lfcc_extractor.extract(test_audio)
    batch = np.expand_dims(features, axis=0)
    
    # Inference
    _ = model.predict(batch, verbose=0)
    
    times.append((time.perf_counter() - start) * 1000)

print(f"Inference times (ms) over {n_iterations} iterations:")
print(f"  Mean: {np.mean(times):.2f}")
print(f"  Std:  {np.std(times):.2f}")
print(f"  Min:  {np.min(times):.2f}")
print(f"  Max:  {np.max(times):.2f}")
print(f"  P95:  {np.percentile(times, 95):.2f}")

if np.mean(times) < 100:
    print("\n[OK] Meets <100ms latency requirement!")
else:
    print("\n[WARN] Does not meet latency requirement")

## 6. Next Steps

1. Download recommended datasets (ASVspoof 2021, LibriSpeech)
2. Organize data into `data/human/` and `data/ai/` directories
3. Run training: `python scripts/train.py --data-dir path/to/data`
4. Convert to TFLite: `python scripts/convert_model.py --model-path saved_models/best_model.keras --quantize`
5. Benchmark: `python scripts/benchmark.py --model-path saved_models/model.tflite`